In [1]:
import sys
from pathlib import Path

# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [2]:
from utils import *
from datasets import load_dataset
from rag import *

In [3]:
gpt_pred = load_dataset("Eathus/github-issues-vul-detection-gpt-few-results-tmp", split="test")
gpt_pred_df = gpt_pred.to_pandas()
gpt_pred_df = gpt_pred_df[~gpt_pred_df.duplicated(subset="issue_github_id", keep=False)]

In [4]:
true_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
false_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & gpt_pred_df.cve_id.isna()]
false_neg = gpt_pred_df[~gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
all_true = gpt_pred_df[~gpt_pred_df.cve_id.isna()]

print("true pos:", len(true_pos))
print("false pos:", len(false_pos))
print("false neg:", len(false_neg))
print("all true:", len(all_true))
print("all", len(gpt_pred_df))

true pos: 296
false pos: 305
false neg: 10
all true: 306
all 1763


In [5]:
cwes = load_dataset("Eathus/cwe_view1000_list_gpt_few_cwe_desc_final_best", split="train")
cwes_df = cwes.to_pandas()

In [6]:
cwes_df.columns

Index(['ID', 'Name', 'Abstraction', 'Structure', 'Status', 'Description',
       'ExtendedDescription', 'ApplicablePlatforms', 'AlternateTerms',
       'ModesOfIntroduction', 'CommonConsequences', 'PotentialMitigations',
       'ObservedExamples', 'AffectedResources', 'TaxonomyMappings',
       'RelatedAttackPatterns', 'References', 'Notes', 'ContentHistory',
       'MappingNotes_Usage', 'MappingNotes_Rationale', 'MappingNotes_Comments',
       'MappingNotes_Reasons', 'MappingNotes_Suggestions', 'RelatedWeaknesses',
       'WeaknessOrdinalities', 'DetectionMethods', 'DemonstrativeExamples',
       'FunctionalAreas', 'Diagram', 'LikelihoodOfExploit',
       'BackgroundDetails', 'NumPaths', 'Paths', 'Children', 'Summary',
       'gpt_cwe_description'],
      dtype='object')

# View CWE-1000

In [7]:
cwes_dict = cwes_df.to_dict('records')
documents_light = [
    create_ordered_cwe_document(weakness, old=True) for weakness in cwes_dict
]
document_dict = {doc.metadata["CWE_ID"]: doc for doc in documents_light}
documents = [
    Document(
        page_content=cwe.gpt_cwe_description,
        metadata={
            "CWE_ID": cwe.ID,  # Original metadata
            #"MappingUsage": cwe.MappingNotes_Usage,
            "Abstraction": CWE_abstraction[cwe.Abstraction.upper()].value,  # New field
            "Child_IDs": cwe.Children.tolist() 
        },  # Original metadata
    )
    for _, cwe in cwes_df.iterrows()
]

#allowed_usage = ["Allowed", "Allowed-with-Review", "Discouraged"]
#documents = filter_cwe_docs_usage(documents, allowed_usage)

documents = split_docs(documents, echo=True, max_tokens=800)

Original: 940 docs
After splitting: 960 docs


In [8]:
documents[0].metadata

{'CWE_ID': '284',
 'Abstraction': 4,
 'Child_IDs': ['1191',
  '1220',
  '1224',
  '1231',
  '1233',
  '1242',
  '1252',
  '1257',
  '1259',
  '1260',
  '1262',
  '1263',
  '1267',
  '1268',
  '1270',
  '1274',
  '1276',
  '1280',
  '1283',
  '1290',
  '1292',
  '1294',
  '1296',
  '1304',
  '1311',
  '1312',
  '1313',
  '1315',
  '1316',
  '1317',
  '1320',
  '1323',
  '1334',
  '269',
  '282',
  '285',
  '286',
  '287',
  '346',
  '749',
  '923'],
 'chunk_id': '284_0',
 'total_chunks': 1,
 'chunk_hash': '0777046ea8d134ddf48e7933ad7e4364',
 'id': 'doc_1'}

In [77]:
def create_vectorstore(
    docs,
    save_path="tmp/faiss_index",
    vec_store="FAISS",
    embedding_model="BAAI/bge-large-en-v1.5",
    device="cuda",
):
    # Embedding is now an argument!
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},
    )

    if vec_store == "FAISS":
        if os.path.exists(f"{save_path}/index.faiss") and os.path.exists(
            f"{save_path}/index.pkl"
        ):
            print("Loading existing vector store...")
            vectorstore = FAISS.load_local(
                folder_path=save_path,
                embeddings=embeddings,
                allow_dangerous_deserialization=True,
            )
        else:
            print(f"Creating new vector store with {embedding_model} ...")
            vectorstore = FAISS.from_documents(docs, embeddings)
            vectorstore.save_local(save_path)
            print(f"Vector store saved to {save_path}")
    elif vec_store == Vectorstore.CHROMADB:
        processed_docs = []
        for doc in docs:
            # Create a copy of the original metadata
            new_metadata = doc.metadata.copy()

            # Only modify Child_IDs if it exists
            new_metadata["Child_IDs"] = ",".join(map(str, new_metadata["Child_IDs"]))

            # Create new Document with same content and updated metadata
            processed_docs.append(
                Document(
                    page_content=doc.page_content,
                    metadata=new_metadata,  # All original metadata preserved
                )
            )
        if os.path.exists(save_path):
            print("Loading existing ChromaDB vector store...")
            vectorstore = Chroma(
                persist_directory=save_path,
                embedding_function=embeddings,
                #collection_name=collection_name,
            )
        else:
            print("Creating new ChromaDB vector store...")
            vectorstore = Chroma.from_documents(
                processed_docs,
                embedding=embeddings,
                persist_directory=save_path,
                #collection_name="index",
            )
            print(f"ChromaDB vector store saved to {save_path}")

    else:
        raise ValueError(f"Unknown vector store type: {vec_store}")

    return vectorstore

def tree_traversal_retrieval(
    query: str,
    vectorstore,
    hierarchy_cache,
    k_list: List = [3, 10, 5, 2, 1],
    top_abstraction=CWE_abstraction.PILLAR,
) -> List[Document]:
    """Perform tree traversal retrieval."""
    k_dict = {
        "PILLAR": k_list[0],
        "CLASS": k_list[1],
        "BASE": k_list[2],
        "VARIANT": k_list[3],
        "COMPOUND": k_list[4],
    }

    def retrieve_level(level, parent_ids=None, depth=0, max_depth=5) -> List[Document]:
        if parent_ids:
            filter_condition = {"CWE_ID": {"$in": parent_ids}}
        else:
            filter_condition = {"Abstraction": {"$eq": level.value}}

        level_query = query
        
        docs = vectorstore.similarity_search(
            query=level_query,
            filter=filter_condition,
            k= k_dict[level.name],
        )
        
        if not docs:
            # print("No docs found, stopping recursion.")
            return []
        if depth > max_depth:
            # print(f"Max depth {max_depth} reached, stopping recursion.")
            return docs

        child_ids = []
        for doc in docs:
            child_ids.extend(
                hierarchy_cache.get(doc.metadata["CWE_ID"], {}).get("children", [])
            )
        child_ids = list(set(child_ids))
        max_level = max([doc.metadata.get("Abstraction", 0) for doc in docs])

        """
        print(
            f"Max level in current docs: {max_level}, number of Child IDs: {len(child_ids)}"
        )
        """

        if not child_ids or max_level == 0:
            # print("No child IDs or max_level=0, stopping recursion.")
            return docs

        child_docs = retrieve_level(
            CWE_abstraction(max_level), parent_ids=child_ids, depth=depth + 1
        )
        return docs + child_docs

    ret = retrieve_level(top_abstraction)

    ret = list({doc.metadata["CWE_ID"]: doc for doc in ret}.values())

    # print(f"Number of documents retrieved: {len(ret)}")
    return ret


def raptor_rag_label(
    data_df,
    vectorstore,
    match_col="gpt_vulnerability",
    k_list: List = [3, 10, 5, 2, 0],
    # max_workers=10,
    top_abstraction=CWE_abstraction.PILLAR,
):  # max for k is 20
    ret_df = data_df.copy()

    hierarchy_cache = {}
    all_metadata = vectorstore.get()["metadatas"]
    for meta in all_metadata:
        hierarchy_cache[meta["CWE_ID"]] = {
            "abstraction": meta["Abstraction"],
            "children": meta["Child_IDs"].split(",") if meta.get("Child_IDs") else [],
        }
    # display(ret_df[ret_df['cve_primary_cwe'].isna()])

    def label(desc):
        try:
            ret = [
                x.metadata["CWE_ID"]
                for x in tree_traversal_retrieval(
                    desc,
                    vectorstore,
                    k_list=k_list,
                    hierarchy_cache=hierarchy_cache,
                    top_abstraction=top_abstraction,
                )
            ]
            # print(len(ret))
            return ret
        except Exception as e:
            print(f"General error processing message: {e}")
            return

    ret_df["rag_candidates"] = ret_df[match_col].progress_map(label)
    ret_df["rag_label"] = ret_df.apply(
        lambda row: (
            row["cve_primary_cwe"]
            if row["cve_primary_cwe"] in row["rag_candidates"]
            else row["rag_candidates"][0]
        ),
        axis=1,
    )

    return ret_df


In [10]:
vectorstore = create_vectorstore(
    documents,
    "tmp/chromadb_raptor_index",
    Vectorstore.CHROMADB,
)

Creating new ChromaDB vector store...
ChromaDB vector store saved to tmp/chromadb_raptor_index


In [81]:
settings = [[9, 9, 10, 1, 1]]
for k_list in settings :
    rag_labeled_df = raptor_rag_label(
        true_pos,
        vectorstore,
        match_col="gpt_vulnerability",
        # device="cpu",
        k_list=k_list,
        top_abstraction=CWE_abstraction.PILLAR,
    )
    setting_str = "-".join(map(str,k_list))
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, retrieval_stats=True, md_filename=f'tmp/raptor_results/classification_report.md')

  0%|          | 0/296 [00:00<?, ?it/s]

100%|██████████| 296/296 [01:00<00:00,  4.90it/s]


✅ Markdown report saved to: tmp/raptor_results/classification_report.md


In [30]:
setting_str = "-".join(map(str, k_list))
print(setting_str)
evaluate_rag(rag_labeled_df, col='rag_label', save=True, retrieval_stats=True, md_filename=f'tmp/raptor_results/classification_report_{setting_str}.md')

4-3-2-1-1

✅ Markdown report saved to: tmp/raptor_results/classification_report_4-3-2-1-1.md


# View CWE-1003

In [67]:
view1003 = load_dataset("Eathus/cwe_view1003_raw_list", split="train")
view1003_df = view1003.to_pandas()

In [107]:
ids_list = view1003_df['ID'].tolist()
documents1003 = [doc for doc in documents if doc.metadata['CWE_ID'] in ids_list]
documents1003 = [
    Document(
        page_content=doc.page_content,
        metadata={
            "CWE_ID": doc.metadata["CWE_ID"],  # Original metadata
            #"MappingUsage": cwe.MappingNotes_Usage,
            "Abstraction": doc.metadata["Abstraction"],  # New field
            "Child_IDs": [child for child in doc.metadata["Child_IDs"] if child in ids_list]
        },  # Original metadata
    )
    for doc in documents1003
]

In [72]:
vectorstore1003 = create_vectorstore(
    documents1003,
    "tmp/chromadb_raptor1003_index",
    Vectorstore.CHROMADB,
)

Creating new ChromaDB vector store...
ChromaDB vector store saved to tmp/chromadb_raptor1003_index


In [117]:
settings = [[2, 8, 2, 2, 2]]
for k_list in settings :
    rag_labeled_df = raptor_rag_label(
        true_pos,
        vectorstore1003,
        match_col="gpt_vulnerability",
        # device="cpu",
        k_list=k_list,
        top_abstraction=CWE_abstraction.CLASS,
    )
    setting_str = "-".join(map(str,k_list))
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, retrieval_stats=True, md_filename=f'tmp/raptor_results/classification_report1003.md')

100%|██████████| 296/296 [00:29<00:00, 10.01it/s]


✅ Markdown report saved to: tmp/raptor_results/classification_report1003.md
